# 02.3 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/02-regression/03-measuring-error/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd
five = pd.read_csv(data_path("shopwave_visits_spend_5.csv"))
five

import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
y = five["spend_next_month"].to_numpy(dtype=float); x = five["visits_last_month"].to_numpy(dtype=float)

## Exercise 1

Residuals: 50−60 = −10, 80−70 = 10, 90−100 = −10, 130−120 = 10, 150−160 = −10. Absolute: all 10 → MAE **10**. Squares: all 100 → MSE **100**, RMSE **10** (equal to MAE because every miss is the same size). $R^2 = 1 - 500/6400 = $ **0.922**. Worse than the line on every metric.

In [2]:
p = np.array([60, 70, 100, 120, 160.])
print(mean_absolute_error(y, p), mean_squared_error(y, p), np.sqrt(mean_squared_error(y, p)), round(r2_score(y, p), 4))

10.0 100.0 10.0 0.9219


## Exercise 2

In [3]:
ks = np.arange(40, 161)
maes = [np.abs(y - k).mean() for k in ks]
mses = [((y - k)**2).mean() for k in ks]
print("MAE-best constant:", ks[np.argmin(maes)], "  median:", np.median(y))
print("MSE-best constant:", ks[np.argmin(mses)], "  mean:  ", y.mean())

MAE-best constant: 90   median: 90.0
MSE-best constant: 100   mean:   100.0


The constant that minimises MAE is the **median**; the one that minimises MSE is the **mean**. This is general: MAE-trained models predict medians, MSE-trained models predict means. For a skewed target that is a real choice.

## Exercise 3

In [4]:
from sklearn.linear_model import LinearRegression
cust20 = pd.read_csv(data_path("shopwave_customers_small.csv"))
y20 = cust20["spend_next_month"]
for col in ["visits_last_month", "items_last_month"]:
    p = LinearRegression().fit(cust20[[col]], y20).predict(cust20[[col]])
    print(f"{col:<18} MAE {mean_absolute_error(y20, p):6.2f}   RMSE {np.sqrt(mean_squared_error(y20, p)):6.2f}")

visits_last_month  MAE  10.65   RMSE  13.48
items_last_month   MAE  19.86   RMSE  24.86


Here the same line wins on both. That is common but not guaranteed; the next exercise shows how one row can split them.

## Exercise 4

In [5]:
line = LinearRegression().fit(cust20[["visits_last_month"]], y20)
X21 = pd.concat([cust20[["visits_last_month"]], pd.DataFrame({"visits_last_month": [3]})], ignore_index=True)
y21 = pd.concat([y20, pd.Series([5000.])], ignore_index=True)
before = {"MAE": mean_absolute_error(y20, line.predict(cust20[["visits_last_month"]])),
          "RMSE": np.sqrt(mean_squared_error(y20, line.predict(cust20[["visits_last_month"]]))),
          "R²": r2_score(y20, line.predict(cust20[["visits_last_month"]]))}
after = {"MAE": mean_absolute_error(y21, line.predict(X21)),
         "RMSE": np.sqrt(mean_squared_error(y21, line.predict(X21))),
         "R²": r2_score(y21, line.predict(X21))}
for k in before:
    print(f"{k:<5} {before[k]:8.2f} → {after[k]:8.2f}   ({(after[k]-before[k])/abs(before[k]):+.0%})")

MAE      10.65 →   244.85   (+2199%)
RMSE     13.48 →  1075.65   (+7879%)
R²        0.96 →    -0.08   (-109%)


One row in twenty-one: MAE rises a lot, RMSE rises far more, and R² falls off a cliff because the squared miss on the wholesaler dwarfs everything else. Decide *before* modelling whether such customers belong in the table at all (01.9's outlier question).